In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [2]:
df = pd.read_csv('Day12_Used_Car_Preprocessing_Dataset.csv')

In [3]:
X = df.drop(columns=['Car_ID', 'Resale_Price_Lakh'])
y = df['Resale_Price_Lakh']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

In [6]:
continuous_outlier_cols = ['Mileage_Km', 'Engine_CC', 'Power_BHP']

for col in continuous_outlier_cols:
    Q1 = X_train_proc[col].quantile(0.25)
    Q3 = X_train_proc[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap training values
    X_train_proc[col] = np.clip(X_train_proc[col], lower_bound, upper_bound)
    # Cap test values using TRAIN bounds to prevent data leakage
    X_test_proc[col] = np.clip(X_test_proc[col], lower_bound, upper_bound)

In [7]:
condition_order = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
ord_enc = OrdinalEncoder(categories=[condition_order])

X_train_proc['Condition_Encoded'] = ord_enc.fit_transform(X_train_proc[['Condition']])
X_test_proc['Condition_Encoded'] = ord_enc.transform(X_test_proc[['Condition']])

X_train_proc.drop(columns=['Condition'], inplace=True)
X_test_proc.drop(columns=['Condition'], inplace=True)

In [8]:
nominal_cols = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']
ohe = OneHotEncoder(drop='first', sparse_output=False)

ohe_train = pd.DataFrame(
    ohe.fit_transform(X_train_proc[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_train_proc.index
)
ohe_test = pd.DataFrame(
    ohe.transform(X_test_proc[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_test_proc.index
)

X_train_proc = X_train_proc.drop(columns=nominal_cols).join(ohe_train)
X_test_proc = X_test_proc.drop(columns=nominal_cols).join(ohe_test)

In [9]:
num_cols = ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
            'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Condition_Encoded']

scaler = StandardScaler()
X_train_proc[num_cols] = scaler.fit_transform(X_train_proc[num_cols])
X_test_proc[num_cols] = scaler.transform(X_test_proc[num_cols])

In [10]:
X_train_proc.to_csv('X_train_processed.csv', index=False)
X_test_proc.to_csv('X_test_processed.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)